# 03 · Join Sofascore + Capology — Turkiye Super Lig 25/26 (snapshot 20260428)

Integración de estadísticas de rendimiento (Sofascore) con datos salariales (Capology)
para la temporada **2025/26 de la Süper Lig turca**.

⚠️ **Nota sobre el snapshot:** la temporada 25/26 está aún en curso. Se trabaja con
una foto fija de Sofascore (`df_turkey_2526_snapshot_20260428.csv`). Este notebook
deberá reejecutarse con los datos definitivos cuando finalice la liga, generando
entonces el master sin sufijo de fecha (`master_turkey_2526.csv`).

**Flujo de matching:**
1. Normalización de nombres (tildes, mayúsculas, caracteres especiales)
2. `TEAM_MAP`: alineación manual de nombres de equipo entre fuentes
3. Merge exacto normalizado
4. Fuzzy matching en cuatro niveles:
   - Score ≥ 0.90 → aceptación automática
   - 0.75 ≤ score < 0.90 → revisión manual
   - 0.50 ≤ score < 0.75 → revisión manual estricta
   - score < 0.50 → revisión manual muy estricta
5. Revisión de jugadores sin salario
6. Guardado en `data/master/`

---

## 1. Imports y rutas

In [1]:
import pandas as pd
import unicodedata
import re
from rapidfuzz import fuzz
from pathlib import Path
from IPython.display import display

ROOT       = Path.cwd().parents[1]
SF_DIR     = ROOT / 'data' / 'processed' / 'sofascore'
CG_DIR     = ROOT / 'data' / 'processed' / 'capology'
MASTER_DIR = ROOT / 'data' / 'master'
MASTER_DIR.mkdir(parents=True, exist_ok=True)

print('✅ Rutas configuradas')
print(f'   Root:   {ROOT}')
print(f'   Master: {MASTER_DIR}')

✅ Rutas configuradas
   Root:   d:\USER\Desktop\TFM
   Master: d:\USER\Desktop\TFM\data\master


## 2. Función de normalización

In [2]:
def normalize(s):
    """
    Normaliza un string para comparación: elimina tildes, pasa a minúsculas,
    elimina caracteres especiales y espacios extra.
    """
    if pd.isna(s):
        return ''
    s = str(s)
    s = unicodedata.normalize('NFKD', s).encode('ascii', 'ignore').decode('ascii')
    s = re.sub(r'[^a-z0-9\s]', ' ', s.lower().strip())
    return re.sub(r'\s+', ' ', s).strip()

print('✅ Función definida')

✅ Función definida


## 3. Carga de datos

In [3]:
df_sf = pd.read_csv(SF_DIR / 'df_turkey_2526_snapshot_20260428.csv').copy()
df_cg = pd.read_csv(CG_DIR / 'cg_turkey_2526.csv').copy()

print(f'Sofascore (snapshot):  {df_sf.shape[0]} jugadores | {df_sf.shape[1]} columnas')
print(f'Capology:              {df_cg.shape[0]} jugadores | {df_cg.shape[1]} columnas')

Sofascore (snapshot):  547 jugadores | 117 columnas
Capology:              526 jugadores | 9 columnas


## 4. Normalización

In [4]:
df_sf['player_norm'] = df_sf['player'].apply(normalize)
df_sf['team_norm']   = df_sf['team'].apply(normalize)
df_cg['player_norm'] = df_cg['player'].apply(normalize)
df_cg['team_norm']   = df_cg['club'].apply(normalize)

print('✅ Normalización aplicada')

✅ Normalización aplicada


## 5. Alineación de equipos (TEAM_MAP)

### 5.1 Identificar discrepancias de nombres de equipo

In [5]:
solo_sf = set(df_sf['team_norm'].unique()) - set(df_cg['team_norm'].unique())
solo_cg = set(df_cg['team_norm'].unique()) - set(df_sf['team_norm'].unique())

print('En Sofascore pero no en Capology:')
for e in sorted(solo_sf): print(f'   {e}')
print()
print('En Capology pero no en Sofascore:')
for e in sorted(solo_cg): print(f'   {e}')

En Sofascore pero no en Capology:
   basaksehir fk
   besiktas jk
   fatih karagumruk
   gaziantep fk
   genclerbirligi
   kasmpasa

En Capology pero no en Sofascore:
   basaksehir
   besiktas
   gaziantep bb
   genclerbirligi sk
   karagumrukspor
   kasimpasa


### 5.2 Aplicar TEAM_MAP

Rellenar con las discrepancias identificadas en la celda anterior.

In [7]:
# ── Ajustar según la celda anterior ──────────────────────────
TEAM_MAP = {'basaksehir':'basaksehir fk',
            'besiktas':'besiktas jk',
            'gaziantep bb':'gaziantep fk',
            'genclerbirligi sk':'genclerbirligi',
            'karagumrukspor':'fatih karagumruk',
            'kasimpasa':'kasmpasa'


}
# ─────────────────────────────────────────────────────────────

df_cg['team_norm'] = df_cg['team_norm'].replace(TEAM_MAP)

diff = set(df_cg['team_norm'].unique()) - set(df_sf['team_norm'].unique())
if diff:
    print(f'⚠️  Equipos de CG aún sin match en SF: {diff}')
else:
    print('✅ Todos los equipos alineados')

✅ Todos los equipos alineados


## 6. Merge exacto normalizado

In [8]:
df_merged = df_sf.merge(
    df_cg[['player_norm', 'team_norm', 'gross_weekly_eur', 'gross_annual_eur',
            'position', 'age', 'nationality']],
    on=['player_norm', 'team_norm'],
    how='left'
)

matched = df_merged['gross_annual_eur'].notna().sum()
total   = len(df_merged)

print(f'Merge exacto: {matched}/{total} ({matched/total:.1%})')
print(f'Sin emparejar: {total - matched}')

Merge exacto: 394/547 (72.0%)
Sin emparejar: 153


## 7. Fuzzy matching sobre los no emparejados

Se generan candidatos para todos los jugadores sin match exacto,
sin umbral mínimo, y se clasifican en cuatro niveles.

In [9]:
df_unmatched = df_merged[df_merged['gross_annual_eur'].isna()].copy()
cg_by_team   = df_cg.groupby('team_norm')['player_norm'].apply(list).to_dict()

rows = []
for _, row in df_unmatched[['player','team','player_norm','team_norm']].drop_duplicates().iterrows():
    candidates = cg_by_team.get(row['team_norm'], [])
    best_match, best_score = None, 0
    for cand in candidates:
        score = fuzz.ratio(row['player_norm'], cand) / 100
        if score > best_score:
            best_score = score
            best_match = cand
    if best_match is not None:
        rows.append({
            'player_sf'  : row['player'],
            'team'       : row['team'],
            'player_norm': row['player_norm'],
            'team_norm'  : row['team_norm'],
            'cg_match'   : best_match,
            'score'      : round(best_score, 3)
        })

df_candidates = pd.DataFrame(rows).sort_values('score', ascending=False)
auto_matches     = df_candidates[df_candidates['score'] >= 0.90].copy()
review_matches   = df_candidates[(df_candidates['score'] >= 0.75) & (df_candidates['score'] < 0.90)].copy()
low_matches      = df_candidates[(df_candidates['score'] >= 0.50) & (df_candidates['score'] < 0.75)].copy()
very_low_matches = df_candidates[df_candidates['score'] < 0.50].copy()

print(f'Auto-aceptados    (score ≥ 0.90):          {len(auto_matches)}')
print(f'Revisión media    (0.75 ≤ score < 0.90):   {len(review_matches)}')
print(f'Revisión estricta (0.50 ≤ score < 0.75):   {len(low_matches)}')
print(f'Revisión muy est. (score < 0.50):           {len(very_low_matches)}')

Auto-aceptados    (score ≥ 0.90):          24
Revisión media    (0.75 ≤ score < 0.90):   20
Revisión estricta (0.50 ≤ score < 0.75):   53
Revisión muy est. (score < 0.50):           56


### 7.1 Matches automáticos (score ≥ 0.90)

Revisar para confirmar que todos son correctos.

In [10]:
auto_matches[['player_sf', 'team', 'cg_match', 'score']]

,player_sf,team,cg_match,score
44,Sofyan Amrabat,Fenerbahçe,sofyan amrabat,1.000
81,Dominik Livaković,Fenerbahçe,dominik livakovic,1.000
3,Abdulkerim Bardakci,Galatasaray,abdulkerim bardakc,0.973
9,Ertugrul Taskiran,Alanyaspor,ertugrul taskran,0.970
22,Kacper Kozłowski,Gaziantep FK,kacper kozlowski,0.968
26,Dimitris Goutas,Gençlerbirliği,dimitrios goutas,0.968
47,Georgi Dzhikiya,Antalyaspor,georgiy dzhikiya,0.968
73,Mateusz Łęgowski,Eyüpspor,mateusz legowski,0.968
66,Taylan Antalyali,Çaykur Rizespor,taylan antalyal,0.968
35,Jakub Kałuziński,Başakşehir FK,jakub kaluzinski,0.968


### 7.2 Revisión media (0.75 ≤ score < 0.90)

Añadir a `EXCLUDE_FROM_FUZZY` el `player_norm` de los incorrectos.

In [11]:
review_matches[['player_sf', 'team', 'cg_match', 'score']]

,player_sf,team,cg_match,score
36,Bahadır Han Güngördü,Konyaspor,bahadr gungordu,0.882
58,Anfernee Jamal Dijksteel,Kocaelispor,anfernee dijksteel,0.857
31,Serginho,Fatih Karagümrük,sergio,0.857
43,Ege Yildirim,Göztepe,ege yldrm,0.857
136,Deniz Eren Dönmezer,Kayserispor,deniz donmezer,0.848
28,Baran Ali Gezek,Eyüpspor,baran gezek,0.846
132,Taylan Utku Aydın,Kasımpaşa,taylan aydn,0.815
103,Habib Gueye,Kasımpaşa,pape habib gueye,0.815
38,Kamil Corekci,Kasımpaşa,kamil ahmet corekci,0.812
77,Tayyip Talha Sanuç,Gaziantep FK,tayyip sanuc,0.800


In [12]:
# ── Falsos positivos a excluir del nivel medio ────────────────
EXCLUDE_FROM_FUZZY = ['ruan'

]
# ─────────────────────────────────────────────────────────────

review_accepted = review_matches[~review_matches['player_norm'].isin(EXCLUDE_FROM_FUZZY)]
print(f'Aceptados: {len(review_accepted)} | Excluidos: {len(EXCLUDE_FROM_FUZZY)}')

Aceptados: 19 | Excluidos: 1


### 7.3 Revisión estricta (0.50 ≤ score < 0.75)

Por defecto ninguno se acepta. Añadir a `ACCEPT_LOW_FUZZY` los correctos.

In [13]:
low_matches[['player_sf', 'team', 'cg_match', 'score']]

,player_sf,team,cg_match,score
59,Amir Murillo,Beşiktaş JK,michael murillo,0.741
27,Eren Elmalı,Galatasaray,evren eren elmali,0.741
96,Mehmet Eray Özbek,Kayserispor,eray ozbek,0.741
78,Burak Bozan,Gaziantep FK,mustafa burak bozan,0.733
54,Muhammet Tunahan Taşçı,Konyaspor,tunahan tasc,0.727
62,Ismail Esat Buga,Konyaspor,esat buga,0.720
7,Barış Alper Yılmaz,Galatasaray,baris yilmaz,0.714
2,Rafa Silva,Beşiktaş JK,jota silva,0.700
100,Gidado Victor Ntino-Emo,Gaziantep FK,victor ntino emo gidado,0.696
86,Michał Nalepa,Gençlerbirliği,emirhan unal,0.667


In [14]:
# ── Matches de score bajo confirmados manualmente ─────────────
ACCEPT_LOW_FUZZY = ['amir murillo',
                    'eren elmal',
                    'mehmet eray ozbek',
                    'burak bozan',
                    'muhammet tunahan tasc',
                    'ismail esat buga',
                    'bars alper ylmaz',
                    'gidado victor ntino emo',
                    'jo jin ho',
                    'peter etebo',
                    'seyfettin anl yasar',
                    'frimpong',
                    'allan'

]
# ─────────────────────────────────────────────────────────────

low_accepted = low_matches[low_matches['player_norm'].isin(ACCEPT_LOW_FUZZY)]
print(f'Aceptados del nivel bajo: {len(low_accepted)}')

Aceptados del nivel bajo: 13


### 7.4 Revisión muy estricta (score < 0.50)

Candidatos con muy baja similitud. Por defecto ninguno se acepta.
Añadir a `ACCEPT_VERY_LOW_FUZZY` los que se confirmen manualmente.

In [15]:
very_low_matches[['player_sf', 'team', 'cg_match', 'score']]

,player_sf,team,cg_match,score
24,David Datro Fofana,Fatih Karagümrük,davide biraschi,0.485
149,Ali Demirbilek,Antalyaspor,berkay topdemir,0.483
46,Enver Kulašin,Gaziantep FK,kacper kozlowski,0.483
141,Gideon Jung,Kayserispor,german onugkha,0.480
106,Tomáš Čvančara,Antalyaspor,kagan arcan,0.480
142,Yusuf Demir,Galatasaray,lucas torreira,0.480
19,Demir Ege Tıknaz,Beşiktaş JK,emir yasar,0.480
21,Robin Yalcin,Eyüpspor,dorin rotariu,0.480
101,Ogulcan Caglayan,Kocaelispor,dan agyei,0.480
112,Andre Gray,Fatih Karagümrük,davide biraschi,0.480


In [16]:
# ── Matches very low confirmados manualmente ──────────────────
ACCEPT_VERY_LOW_FUZZY = ['marius mouandilmadji',
                         'hwang ui jo'

]
# ─────────────────────────────────────────────────────────────

very_low_accepted = very_low_matches[very_low_matches['player_norm'].isin(ACCEPT_VERY_LOW_FUZZY)]
print(f'Aceptados del nivel very low: {len(very_low_accepted)}')

Aceptados del nivel very low: 2


### 7.5 Aplicar todos los fuzzy matches aceptados

In [17]:
all_fuzzy    = pd.concat([auto_matches, review_accepted, low_accepted, very_low_accepted], ignore_index=True)
fuzzy_lookup = dict(zip(all_fuzzy['player_norm'], all_fuzzy['cg_match']))

df_merged['player_norm_fuzzy'] = df_merged.apply(
    lambda r: fuzzy_lookup.get(r['player_norm'], r['player_norm'])
    if pd.isna(r['gross_annual_eur']) else r['player_norm'],
    axis=1
)

df_final = (
    df_merged
    .drop(columns=['gross_weekly_eur', 'gross_annual_eur', 'position', 'age', 'nationality'])
    .merge(
        df_cg[['player_norm', 'team_norm', 'gross_weekly_eur', 'gross_annual_eur',
               'position', 'age', 'nationality']],
        left_on=['player_norm_fuzzy', 'team_norm'],
        right_on=['player_norm', 'team_norm'],
        how='left'
    )
    .drop(columns=['player_norm_y', 'player_norm_fuzzy'])
    .rename(columns={'player_norm_x': 'player_norm'})
)

matched_final = df_final['gross_annual_eur'].notna().sum()
print(f'Resultado final: {matched_final}/{len(df_final)} ({matched_final/len(df_final):.1%})')
print(f'Sin salario:     {len(df_final) - matched_final}')

Resultado final: 450/547 (82.3%)
Sin salario:     97


## 8. Revisión de jugadores sin salario

Ordenados por equipo y minutos jugados para identificar si alguno debería tener salario.

In [18]:
sin_salario = (
    df_final[df_final['gross_annual_eur'].isna()]
    [['player', 'team', 'minutesPlayed', 'appearances', 'goals', 'assists']]
    .sort_values(['team', 'minutesPlayed'], ascending=[True, False])
    .reset_index(drop=True)
)

pd.set_option('display.max_rows', None)
print(f'Total sin salario: {len(sin_salario)}')
display(sin_salario)
pd.reset_option('display.max_rows')

Total sin salario: 97


,player,team,minutesPlayed,appearances,goals,assists
0,Ruan Pereira Duarte,Alanyaspor,2072,26,1,1
1,Uchenna Ogundu,Alanyaspor,993,17,2,1
2,Andraž Šporar,Alanyaspor,165,2,0,0
3,Tomáš Čvančara,Antalyaspor,533,11,1,0
4,Güray Vural,Antalyaspor,277,4,2,0
5,Poyraz Efe Yıldırım,Antalyaspor,180,10,0,0
6,Mert Yilmaz,Antalyaspor,36,3,0,0
7,Ali Demirbilek,Antalyaspor,1,1,0,0
8,Deniz Dilmen,Başakşehir FK,90,1,0,0
9,Ömer Faruk Beyaz,Başakşehir FK,30,3,0,0


### 8.1 Comparación manual por equipo

Para cada equipo con jugadores sin salario se muestra la plantilla completa de Capology
ordenada alfabéticamente por nombre normalizado, facilitando la detección visual de matches fallidos.

In [19]:
equipos_sin_salario = sin_salario['team'].unique()

for equipo in sorted(equipos_sin_salario):
    sf_jugadores = sin_salario[sin_salario['team'] == equipo][['player', 'minutesPlayed']].sort_values('player')

    equipo_norm  = normalize(equipo)
    cg_jugadores = (
        df_cg[df_cg['team_norm'] == equipo_norm][['player', 'player_norm']]
        .sort_values('player_norm')
        .reset_index(drop=True)
    )

    print(f'\n{"="*60}')
    print(f'  {equipo}  —  SF sin salario:')
    display(sf_jugadores.reset_index(drop=True))
    print(f'  CG plantilla completa:')
    display(cg_jugadores)


  Alanyaspor  —  SF sin salario:


,player,minutesPlayed
0,Andraž Šporar,165
1,Ruan Pereira Duarte,2072
2,Uchenna Ogundu,993


  CG plantilla completa:


,player,player_norm
0,Baran Moğultay,baran mogultay
1,Batuhan Yavuz,batuhan yavuz
2,Bruno Viana,bruno viana
3,Buluthan Bulut,buluthan bulut
4,Efecan Karaca,efecan karaca
5,Enes Keskin,enes keskin
6,Ertuğrul Taşkıran,ertugrul taskran
7,Fatih Aksoy,fatih aksoy
8,Fidan Aliti,fidan aliti
9,Florent Hadergjonaj,florent hadergjonaj



  Antalyaspor  —  SF sin salario:


...

,player,player_norm
0,Ahmet Yıldırım,ahmet yldrm
1,André Onana,andre onana
2,Anthony Nwakaeme,anthony nwakaeme
3,Arda Öztürk,arda ozturk
4,Arseniy Batagov,arseniy batagov
5,Benjamin Bouchouari,benjamin bouchouari
6,Boran Başkan,boran baskan
7,Chibuike Nwaiwu,chibuike nwaiwu
8,Christ Inao Oulaï,christ inao oulai
9,Cihan Çanak,cihan canak



  Çaykur Rizespor  —  SF sin salario:


,player,minutesPlayed
0,Jesurun Rak-Sakyi,817
1,Vaclav Jurecka,479


  CG plantilla completa:


,player,player_norm
0,Adedire Mebude,adedire mebude
1,Ali Sowe,ali sowe
2,Altin Zeqiri,altin zeqiri
3,Attila Mocsi,attila mocsi
4,Casper Højer,casper hjer
5,Efe Doğan,efe dogan
6,Emir Ortakaya,emir ortakaya
7,Emrecan Bulut,emrecan bulut
8,Erdem Canpolat,erdem canpolat
9,Frantzdy Pierrot,frantzdy pierrot


In [20]:
# ── Matches manuales (nombres muy distintos o traspasos invernales) ──
# Formato: (player_norm_sf, team_norm_sf): (player_norm_cg, team_norm_cg)
MANUAL_MATCHES = {
    ('ruan pereira duarte', 'alanyaspor'): ('ruan',  'alanyaspor'),
    ('juan santos da silva', 'goztepe')  : ('juan',  'goztepe'),
}
# ────────────────────────────────────────────────────────────────────
print(f'Matches manuales definidos: {len(MANUAL_MATCHES)}')

Matches manuales definidos: 2


In [21]:
# Aplicar matches manuales sobre los que siguen sin salario
for (p_sf, t_sf), (p_cg, t_cg) in MANUAL_MATCHES.items():
    mask = (df_final['player_norm'] == p_sf) & (df_final['team_norm'] == t_sf) & (df_final['gross_annual_eur'].isna())
    datos_cg = df_cg[(df_cg['player_norm'] == p_cg) & (df_cg['team_norm'] == t_cg)]
    if not datos_cg.empty and mask.any():
        for col in ['gross_weekly_eur', 'gross_annual_eur', 'position', 'age', 'nationality']:
            df_final.loc[mask, col] = datos_cg[col].values[0]
        print(f'✅ Match manual aplicado: {p_sf} ({t_sf}) → {p_cg} ({t_cg})')
    else:
        print(f'⚠️  No encontrado: {p_sf} ({t_sf}) → {p_cg} ({t_cg})')

matched_tras_manual = df_final['gross_annual_eur'].notna().sum()
print(f'\nTras matches manuales: {matched_tras_manual}/{len(df_final)} ({matched_tras_manual/len(df_final):.1%})')

✅ Match manual aplicado: ruan pereira duarte (alanyaspor) → ruan (alanyaspor)
✅ Match manual aplicado: juan santos da silva (goztepe) → juan (goztepe)

Tras matches manuales: 452/547 (82.6%)


## 9. Guardado

Una vez revisado todo, se eliminan las columnas auxiliares y se guarda en `data/master/`.

⚠️ Nombre con sufijo `_snapshot_20260428` para diferenciar del master definitivo
que se generará al cierre de la temporada (`master_turkey_2526.csv`).

In [22]:
df_final = df_final.drop(columns=['player_norm', 'team_norm'])

nombre_salida = 'master_turkey_2526_snapshot_20260428.csv'
df_final.to_csv(MASTER_DIR / nombre_salida, index=False)

print(f'✅ Guardado: {nombre_salida}')
print(f'   Jugadores totales:  {len(df_final)}')
print(f'   Con salario:        {df_final["gross_annual_eur"].notna().sum()}')
print(f'   Sin salario (NaN):  {df_final["gross_annual_eur"].isna().sum()}')
print(f'   Columnas:           {df_final.shape[1]}')

✅ Guardado: master_turkey_2526_snapshot_20260428.csv
   Jugadores totales:  547
   Con salario:        452
   Sin salario (NaN):  95
   Columnas:           122
